# Multi-Agent Real Estate and Mortgage Analysis System

This notebook implements a multi-agent, multi-modal architecture for real estate analysis and mortgage advisory:

- **Multi-agent architecture** with specialized agents for property scanning, valuation, mortgage analysis, and reporting
- **Real-time Gradio UI** with threading and queues for live agent activity logs
- **Multi-modal inputs** supporting property images and text descriptions
- **Mortgage calculator agent** analyzing loan scenarios, affordability, and rate comparisons
- **Valuation ensemble agent** combining rule-based estimates with LLM-powered analysis
- **RAG-style memory** persisting property records and analysis across sessions
- **3D vector space visualization** of property embeddings by type, price, and location

In [ ]:
%pip install -q gradio pydantic openai chromadb sentence-transformers scikit-learn plotly pillow requests numpy

In [ ]:
import os
import logging
import queue
import threading
import time
import json
import base64
import math
from typing import List, Optional, Dict, Any
from datetime import datetime
from io import BytesIO

import gradio as gr
import numpy as np
import plotly.graph_objects as go
from pydantic import BaseModel, Field

## Data Models

Pydantic models representing the core domain objects passed between agents.

In [ ]:
class Property(BaseModel):
    address: str
    property_type: str  # residential, condo, commercial, land
    bedrooms: int
    bathrooms: float
    sqft: int
    listing_price: float
    year_built: int
    neighborhood: str
    description: str
    url: str
    image_path: Optional[str] = None


class PropertySelection(BaseModel):
    properties: List[Property]


class MortgageScenario(BaseModel):
    loan_amount: float
    down_payment_pct: float
    interest_rate: float
    term_years: int
    monthly_payment: float
    total_interest: float
    affordability_score: float  # 0-100


class PropertyOpportunity(BaseModel):
    property: Property
    estimated_value: float
    value_gap: float  # positive = underpriced
    investment_score: float  # 0-100
    mortgage: MortgageScenario
    analysis_summary: str
    timestamp: str = Field(default_factory=lambda: datetime.now().isoformat())

## Logging Infrastructure

Queue-based logging handler streams agent activity to the Gradio UI in real time.

In [ ]:
BG_BLACK  = '\033[40m'
RED       = '\033[31m'
GREEN     = '\033[32m'
YELLOW    = '\033[33m'
BLUE      = '\033[34m'
MAGENTA   = '\033[35m'
CYAN      = '\033[36m'
WHITE     = '\033[37m'
BG_BLUE   = '\033[44m'
RESET     = '\033[0m'

color_mapper = {
    BG_BLACK + RED:     "#dd4444",
    BG_BLACK + GREEN:   "#44cc44",
    BG_BLACK + YELLOW:  "#dddd00",
    BG_BLACK + BLUE:    "#5588ee",
    BG_BLACK + MAGENTA: "#bb44dd",
    BG_BLACK + CYAN:    "#00cccc",
    BG_BLACK + WHITE:   "#aaddff",
    BG_BLUE  + WHITE:   "#ff8822",
}


def reformat_log(message: str) -> str:
    for key, value in color_mapper.items():
        message = message.replace(key, f'<span style="color: {value}">')
    message = message.replace(RESET, '</span>')
    return message


class QueueHandler(logging.Handler):
    def __init__(self, log_queue: queue.Queue):
        super().__init__()
        self.log_queue = log_queue

    def emit(self, record):
        self.log_queue.put(self.format(record))


def setup_logging(log_queue: queue.Queue):
    handler = QueueHandler(log_queue)
    formatter = logging.Formatter(
        "[%(asctime)s] %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )
    handler.setFormatter(formatter)
    logger = logging.getLogger()
    for h in logger.handlers[:]:
        logger.removeHandler(h)
    logger.addHandler(handler)
    logger.setLevel(logging.INFO)


def html_for(log_data: List[str]) -> str:
    output = '<br>'.join(log_data[-25:])
    return f"""
    <div style="height:440px; overflow-y:auto; border:1px solid #333; background:#0f1117;
                padding:14px; font-family:monospace; font-size:12px; color:#ccc;">
        {output}
    </div>
    """

## Specialized Agents

Each agent has a single, well-defined responsibility. Agents log their activity and can be swapped for API-backed implementations.

In [ ]:
class BaseAgent:
    name: str = "Base Agent"
    color: str = BG_BLACK + WHITE

    def log(self, message: str):
        logging.info(f"{self.color}[{self.name}]{RESET} {message}")


# ---------------------------------------------------------------------------
# Property Scanner Agent
# Simulates fetching listings from MLS feeds, Zillow, or Redfin APIs.
# In production: replace with real API calls or web scraping.
# ---------------------------------------------------------------------------
class PropertyScannerAgent(BaseAgent):
    name = "PropertyScanner"
    color = BG_BLACK + CYAN

    MOCK_LISTINGS = [
        Property(
            address="4821 Maple Grove Drive, Austin, TX 78745",
            property_type="residential",
            bedrooms=3, bathrooms=2.0, sqft=1850,
            listing_price=389_000,
            year_built=2004,
            neighborhood="South Austin",
            description="Updated 3/2 in desirable South Austin. Open floor plan, granite counters, hardwood floors, two-car garage. Close to Slaughter Lane retail corridor.",
            url="https://example.com/listings/4821-maple-grove"
        ),
        Property(
            address="1130 W 5th Street Unit 204, Austin, TX 78703",
            property_type="condo",
            bedrooms=1, bathrooms=1.0, sqft=780,
            listing_price=295_000,
            year_built=2018,
            neighborhood="Downtown / Clarksville",
            description="Modern condo in walkable Clarksville. Stainless appliances, quartz counters, rooftop terrace access, dedicated parking. High rental demand area.",
            url="https://example.com/listings/1130-w5th-204"
        ),
        Property(
            address="9302 Buttercup Lane, Round Rock, TX 78681",
            property_type="residential",
            bedrooms=4, bathrooms=3.0, sqft=2640,
            listing_price=472_000,
            year_built=2015,
            neighborhood="Stone Canyon",
            description="Spacious family home in Stone Canyon. Primary suite downstairs, game room up, covered patio, community pool. Excellent Round Rock ISD schools.",
            url="https://example.com/listings/9302-buttercup"
        ),
        Property(
            address="2200 E Cesar Chavez St, Austin, TX 78702",
            property_type="commercial",
            bedrooms=0, bathrooms=2.0, sqft=3100,
            listing_price=890_000,
            year_built=1978,
            neighborhood="East Austin",
            description="Mixed-use commercial building on high-traffic East Cesar Chavez corridor. Currently retail on ground floor, storage upstairs. Redevelopment potential.",
            url="https://example.com/listings/2200-chavez"
        ),
        Property(
            address="7714 Scenic Bluff Drive, Austin, TX 78730",
            property_type="residential",
            bedrooms=5, bathrooms=4.5, sqft=4200,
            listing_price=1_150_000,
            year_built=2008,
            neighborhood="River Place",
            description="Luxury Hill Country estate with canyon views. Chef kitchen, wine cellar, resort pool and spa, three-car garage, home theater. Gated community.",
            url="https://example.com/listings/7714-scenic-bluff"
        ),
    ]

    def scan(self, memory: List[PropertyOpportunity] = None) -> PropertySelection:
        self.log("Scanning MLS feeds and listing APIs for new properties")
        time.sleep(1.2)

        seen_addresses = set()
        if memory:
            seen_addresses = {opp.property.address for opp in memory}
            self.log(f"Excluding {len(seen_addresses)} previously analyzed properties from memory")

        new_listings = [p for p in self.MOCK_LISTINGS if p.address not in seen_addresses]
        self.log(f"Found {len(new_listings)} new listings to analyze")
        return PropertySelection(properties=new_listings)


# ---------------------------------------------------------------------------
# Valuation Agent
# Estimates fair market value using comps, price-per-sqft heuristics, and
# condition/location adjustments. In production: integrate an AVM API or
# a fine-tuned regression model.
# ---------------------------------------------------------------------------
class ValuationAgent(BaseAgent):
    name = "ValuationAgent"
    color = BG_BLACK + GREEN

    BASE_PRICE_PER_SQFT = {
        "residential": 210,
        "condo":       280,
        "commercial":  195,
        "land":        85,
    }

    NEIGHBORHOOD_MULTIPLIER = {
        "Downtown / Clarksville": 1.40,
        "East Austin":            1.25,
        "South Austin":           1.10,
        "River Place":            1.35,
        "Stone Canyon":           1.05,
        "default":                1.00,
    }

    def estimate(self, prop: Property) -> float:
        self.log(f"Estimating fair market value for: {prop.address}")
        time.sleep(0.6)

        base_ppsf = self.BASE_PRICE_PER_SQFT.get(prop.property_type, 200)
        neighborhood_mult = self.NEIGHBORHOOD_MULTIPLIER.get(
            prop.neighborhood, self.NEIGHBORHOOD_MULTIPLIER["default"]
        )

        # Age discount: 0.25% per year over 10 years old, capped at 15%
        age = datetime.now().year - prop.year_built
        age_discount = min(max(age - 10, 0) * 0.0025, 0.15)

        # Bedroom/bathroom premium for residential
        bed_premium = 0
        if prop.property_type in ("residential", "condo"):
            bed_premium = (prop.bedrooms * 8_000) + (prop.bathrooms * 5_000)

        base_value = (prop.sqft * base_ppsf * neighborhood_mult * (1 - age_discount)) + bed_premium
        self.log(f"Estimated value: ${base_value:,.0f} (listing: ${prop.listing_price:,.0f})")
        return round(base_value, 2)


# ---------------------------------------------------------------------------
# Mortgage Agent
# Computes monthly payments, total interest, and an affordability score.
# Produces multiple scenarios (20%, 10%, 5% down) for comparison.
# In production: pull live rates from a mortgage rate API.
# ---------------------------------------------------------------------------
class MortgageAgent(BaseAgent):
    name = "MortgageAgent"
    color = BG_BLACK + YELLOW

    CURRENT_30Y_RATE = 0.0688   # 6.88% - update or pull from API in production
    CURRENT_15Y_RATE = 0.0619
    MEDIAN_US_INCOME = 75_000

    def _monthly_payment(self, principal: float, annual_rate: float, term_years: int) -> float:
        if annual_rate == 0:
            return principal / (term_years * 12)
        r = annual_rate / 12
        n = term_years * 12
        return principal * (r * (1 + r) ** n) / ((1 + r) ** n - 1)

    def analyze(self, prop: Property, down_payment_pct: float = 0.20, term_years: int = 30) -> MortgageScenario:
        self.log(f"Running mortgage analysis: {down_payment_pct*100:.0f}% down, {term_years}-year fixed")
        time.sleep(0.4)

        rate = self.CURRENT_30Y_RATE if term_years == 30 else self.CURRENT_15Y_RATE
        down = prop.listing_price * down_payment_pct
        loan = prop.listing_price - down
        monthly = self._monthly_payment(loan, rate, term_years)
        total_interest = (monthly * term_years * 12) - loan

        # Affordability score: assume 28% front-end DTI rule on median income
        required_annual_income = monthly * 12 / 0.28
        affordability_score = max(0, min(100, (self.MEDIAN_US_INCOME / required_annual_income) * 100))

        self.log(f"Monthly payment: ${monthly:,.2f} | Total interest: ${total_interest:,.0f}")
        return MortgageScenario(
            loan_amount=round(loan, 2),
            down_payment_pct=down_payment_pct,
            interest_rate=rate,
            term_years=term_years,
            monthly_payment=round(monthly, 2),
            total_interest=round(total_interest, 2),
            affordability_score=round(affordability_score, 1),
        )


# ---------------------------------------------------------------------------
# Image Analysis Agent
# In production: sends property images to a vision model (GPT-4o, Claude)
# to assess condition, curb appeal, and notable features.
# ---------------------------------------------------------------------------
class ImageAnalysisAgent(BaseAgent):
    name = "ImageAnalysis"
    color = BG_BLACK + MAGENTA

    def analyze_image(self, image_path: Optional[str], prop: Property) -> str:
        if not image_path:
            self.log("No image provided; skipping visual analysis")
            return "No image available for visual analysis."

        self.log(f"Analyzing property image for: {prop.address}")
        time.sleep(0.8)

        # In production, this would call a vision API:
        #   with open(image_path, 'rb') as f:
        #       b64 = base64.b64encode(f.read()).decode()
        #   response = openai_client.chat.completions.create(
        #       model="gpt-4o",
        #       messages=[{"role": "user", "content": [
        #           {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
        #           {"type": "text", "text": "Assess this property image..."}]}])
        # )

        # Mock vision output based on property type
        assessments = {
            "residential": "Image shows well-maintained exterior with good curb appeal. Landscaping is tidy. Roof appears in good condition. No visible deferred maintenance.",
            "condo": "Modern unit with clean finishes visible. Building exterior is contemporary and well-kept. Common areas appear maintained.",
            "commercial": "Commercial frontage shows moderate wear. Signage removed indicating vacancy. Structural elements appear sound but cosmetic updates likely needed.",
            "land": "Raw land parcel. Relatively flat with some vegetation. Utilities may need to be extended to site.",
        }
        result = assessments.get(prop.property_type, "Property image analyzed. Condition appears average.")
        self.log("Visual analysis complete")
        return result


# ---------------------------------------------------------------------------
# Reporting / Messaging Agent
# Composes a natural-language summary combining all agent outputs.
# In production: could send SMS (Twilio), email, or push notification.
# ---------------------------------------------------------------------------
class ReportingAgent(BaseAgent):
    name = "ReportingAgent"
    color = BG_BLUE + WHITE

    def compose_summary(self, opp: PropertyOpportunity, vision_notes: str) -> str:
        self.log(f"Composing analysis report for: {opp.property.address}")
        direction = "underpriced" if opp.value_gap > 0 else "overpriced"
        gap_pct = abs(opp.value_gap) / opp.estimated_value * 100

        summary = (
            f"{opp.property.address} | {opp.property.property_type.title()} | "
            f"{opp.property.bedrooms}bd/{opp.property.bathrooms}ba | "
            f"{opp.property.sqft:,} sqft | Built {opp.property.year_built}\n"
            f"Listing Price: ${opp.property.listing_price:,.0f} | "
            f"Estimated Value: ${opp.estimated_value:,.0f} | "
            f"Gap: {direction} by {gap_pct:.1f}% (${abs(opp.value_gap):,.0f})\n"
            f"Investment Score: {opp.investment_score:.0f}/100\n"
            f"Mortgage (20% down, 30yr): ${opp.mortgage.monthly_payment:,.2f}/mo | "
            f"Rate: {opp.mortgage.interest_rate*100:.2f}% | "
            f"Affordability: {opp.mortgage.affordability_score:.0f}/100\n"
            f"Vision: {vision_notes}"
        )
        self.log(f"Report ready | Investment score: {opp.investment_score:.0f}")
        return summary

    def alert(self, opp: PropertyOpportunity):
        self.log(
            f"HIGH-VALUE ALERT: {opp.property.address} | "
            f"Score {opp.investment_score:.0f}/100 | "
            f"Value gap ${opp.value_gap:,.0f}"
        )

## Planning Agent

Orchestrates the full analysis pipeline: scan -> value -> mortgage -> image -> report. Applies a scoring heuristic and triggers alerts for high-opportunity properties.

In [ ]:
class PlanningAgent(BaseAgent):
    name = "PlanningAgent"
    color = BG_BLACK + BLUE

    ALERT_SCORE_THRESHOLD = 60

    def __init__(self):
        self.scanner   = PropertyScannerAgent()
        self.valuator  = ValuationAgent()
        self.mortgage  = MortgageAgent()
        self.vision    = ImageAnalysisAgent()
        self.reporter  = ReportingAgent()

    def _investment_score(self, prop: Property, estimated_value: float, mortgage: MortgageScenario) -> float:
        """Composite investment score 0-100."""
        value_gap     = estimated_value - prop.listing_price
        gap_score     = min(max((value_gap / estimated_value) * 200, -50), 50)  # -50 to +50
        afford_score  = mortgage.affordability_score * 0.3                       # 0 to 30
        age_score     = max(0, 20 - (datetime.now().year - prop.year_built) * 0.4)  # 0 to 20
        return round(min(max(50 + gap_score + afford_score * 0 + age_score, 0), 100), 1)

    def plan(self, memory: List[PropertyOpportunity] = None) -> List[PropertyOpportunity]:
        if memory is None:
            memory = []

        self.log("Starting property analysis cycle")

        selection = self.scanner.scan(memory)
        if not selection or not selection.properties:
            self.log("No new properties to analyze")
            return []

        new_opportunities = []

        for prop in selection.properties:
            self.log(f"Analyzing property: {prop.address}")

            estimated_value = self.valuator.estimate(prop)
            mortgage        = self.mortgage.analyze(prop)
            vision_notes    = self.vision.analyze_image(prop.image_path, prop)
            value_gap       = estimated_value - prop.listing_price
            inv_score       = self._investment_score(prop, estimated_value, mortgage)

            opp = PropertyOpportunity(
                property=prop,
                estimated_value=estimated_value,
                value_gap=value_gap,
                investment_score=inv_score,
                mortgage=mortgage,
                analysis_summary="",
            )

            opp.analysis_summary = self.reporter.compose_summary(opp, vision_notes)

            if inv_score >= self.ALERT_SCORE_THRESHOLD:
                self.reporter.alert(opp)

            new_opportunities.append(opp)
            self.log(f"Analysis complete for: {prop.address} | Score: {inv_score}")

        new_opportunities.sort(key=lambda x: x.investment_score, reverse=True)
        self.log(f"Cycle complete. Analyzed {len(new_opportunities)} properties.")
        return new_opportunities

## Agent Framework

Top-level coordinator managing persistent memory across analysis runs.

In [ ]:
class RealEstateAgentFramework:
    MEMORY_FILE = "real_estate_memory.json"

    def __init__(self):
        self.memory:  List[PropertyOpportunity] = self._read_memory()
        self.planner: Optional[PlanningAgent]   = None

    def init_agents(self):
        if not self.planner:
            logging.info("Initializing Real Estate Agent Framework")
            self.planner = PlanningAgent()
            logging.info("All agents initialized and ready")

    def _read_memory(self) -> List[PropertyOpportunity]:
        if os.path.exists(self.MEMORY_FILE):
            try:
                with open(self.MEMORY_FILE, 'r') as f:
                    data = json.load(f)
                return [PropertyOpportunity(**item) for item in data]
            except Exception:
                return []
        return []

    def _write_memory(self):
        data = [opp.dict() for opp in self.memory]
        with open(self.MEMORY_FILE, 'w') as f:
            json.dump(data, f, indent=2)

    def run(self) -> List[PropertyOpportunity]:
        self.init_agents()
        new_opps = self.planner.plan(memory=self.memory)

        if new_opps:
            self.memory.extend(new_opps)
            self._write_memory()

        return new_opps

    @classmethod
    def get_plot_data(cls, opportunities: List[PropertyOpportunity]):
        """Return coordinate and label data for the 3D vector space plot."""
        if not opportunities:
            n = 60
            vectors = np.random.randn(n, 3)
            labels  = [f"Property {i}" for i in range(n)]
            colors  = ["#5588ee"] * n
            sizes   = [4] * n
            return labels, vectors, colors, sizes

        type_colors = {
            "residential": "#44cc44",
            "condo":       "#5588ee",
            "commercial":  "#ff8822",
            "land":        "#bb44dd",
        }

        labels, x_vals, y_vals, z_vals, colors, sizes = [], [], [], [], [], []

        for opp in opportunities:
            p = opp.property
            x_vals.append(p.listing_price / 1_000_000)          # price axis
            y_vals.append(p.sqft / 1_000)                       # size axis
            z_vals.append(opp.investment_score / 100)           # score axis
            labels.append(p.address.split(',')[0])
            colors.append(type_colors.get(p.property_type, "#aaddff"))
            sizes.append(6 + opp.investment_score / 10)

        vectors = np.column_stack([x_vals, y_vals, z_vals])
        return labels, vectors, colors, sizes

## Visualization

Interactive 3D scatter plot mapping properties by price, size, and investment score.

In [ ]:
def get_plot(framework: Optional[RealEstateAgentFramework] = None) -> go.Figure:
    try:
        opps = framework.memory if framework else []
        labels, vectors, colors, sizes = RealEstateAgentFramework.get_plot_data(opps)

        hover_text = [
            f"{labels[i]}<br>Price: ${vectors[i,0]*1e6:,.0f}<br>Sqft: {vectors[i,1]*1000:,.0f}<br>Score: {vectors[i,2]*100:.0f}"
            for i in range(len(labels))
        ] if opps else [f"Point {i}" for i in range(len(labels))]

        fig = go.Figure(data=[go.Scatter3d(
            x=vectors[:, 0],
            y=vectors[:, 1],
            z=vectors[:, 2],
            mode='markers',
            marker=dict(size=sizes, color=colors, opacity=0.85, line=dict(width=0.5, color='#222')),
            text=hover_text,
            hoverinfo='text',
        )])

        fig.update_layout(
            scene=dict(
                xaxis_title='Price ($M)',
                yaxis_title='Size (k sqft)',
                zaxis_title='Investment Score',
                xaxis=dict(backgroundcolor='#111', gridcolor='#333', zerolinecolor='#555'),
                yaxis=dict(backgroundcolor='#111', gridcolor='#333', zerolinecolor='#555'),
                zaxis=dict(backgroundcolor='#111', gridcolor='#333', zerolinecolor='#555'),
                aspectmode='manual',
                aspectratio=dict(x=2.2, y=2.2, z=1),
                camera=dict(eye=dict(x=1.6, y=1.6, z=0.8)),
                bgcolor='#0f1117',
            ),
            height=440,
            margin=dict(r=5, b=5, l=5, t=5),
            paper_bgcolor='#0f1117',
            plot_bgcolor='#0f1117',
            font=dict(color='#cccccc'),
            title=dict(
                text="Property Space: Price / Size / Investment Score",
                font=dict(color='#aaaaaa', size=13),
                x=0.5
            )
        )
        return fig

    except Exception as e:
        fig = go.Figure()
        fig.update_layout(
            title=f'Plot error: {str(e)}',
            height=440,
            paper_bgcolor='#0f1117',
            font=dict(color='#cccccc')
        )
        return fig

## Mortgage Calculator Utility

Standalone function powering the interactive mortgage tab in the UI.

In [ ]:
def calculate_mortgage_scenarios(home_price: float, annual_income: float) -> str:
    """Return an HTML comparison table for three down payment scenarios."""
    RATE_30 = 0.0688
    RATE_15 = 0.0619

    def monthly(principal, rate, years):
        r, n = rate / 12, years * 12
        return principal * (r * (1 + r) ** n) / ((1 + r) ** n - 1)

    scenarios = [
        ("20% Down", 0.20, 30, RATE_30),
        ("10% Down", 0.10, 30, RATE_30),
        ("5% Down",  0.05, 30, RATE_30),
        ("20% Down", 0.20, 15, RATE_15),
    ]

    rows = ""
    for label, dp, years, rate in scenarios:
        down  = home_price * dp
        loan  = home_price - down
        pmt   = monthly(loan, rate, years)
        total = pmt * years * 12
        interest = total - loan
        dti   = (pmt * 12) / annual_income * 100 if annual_income > 0 else 0
        dti_color = "#44cc44" if dti <= 28 else ("#dddd00" if dti <= 36 else "#dd4444")
        rows += (
            f"<tr>"
            f"<td>{label} / {years}yr</td>"
            f"<td>${down:,.0f}</td>"
            f"<td>${loan:,.0f}</td>"
            f"<td>{rate*100:.2f}%</td>"
            f"<td><b>${pmt:,.2f}</b></td>"
            f"<td>${interest:,.0f}</td>"
            f"<td style='color:{dti_color}'>{dti:.1f}%</td>"
            f"</tr>"
        )

    return f"""
    <div style="background:#0f1117; padding:16px; border-radius:8px; color:#ccc; font-family:monospace; font-size:13px;">
        <div style="margin-bottom:10px; font-size:14px; color:#aaddff;">
            Home Price: <b>${home_price:,.0f}</b> | Annual Income: <b>${annual_income:,.0f}</b>
        </div>
        <table style="width:100%; border-collapse:collapse;">
            <thead>
                <tr style="color:#888; border-bottom:1px solid #333;">
                    <th align="left">Scenario</th>
                    <th align="left">Down Pmt</th>
                    <th align="left">Loan Amt</th>
                    <th align="left">Rate</th>
                    <th align="left">Monthly</th>
                    <th align="left">Total Interest</th>
                    <th align="left">DTI Ratio</th>
                </tr>
            </thead>
            <tbody>
                {rows}
            </tbody>
        </table>
        <div style="margin-top:10px; font-size:11px; color:#666;">
            DTI color guide: green = below 28% (ideal), yellow = 28-36%, red = above 36% (caution).
            Rates are illustrative; verify current rates with your lender.
        </div>
    </div>
    """

## Gradio Application

Multi-tab UI with:
- **Property Scanner**: live agent activity log, results table, 3D property space
- **Mortgage Calculator**: interactive down payment and income scenario tool
- **Property Detail**: full analysis summary for a selected property

In [ ]:
class RealEstateApp:

    def __init__(self):
        self.framework: Optional[RealEstateAgentFramework] = None

    def get_framework(self) -> RealEstateAgentFramework:
        if not self.framework:
            self.framework = RealEstateAgentFramework()
            self.framework.init_agents()
        return self.framework

    # ------------------------------------------------------------------
    # Data helpers
    # ------------------------------------------------------------------
    def opportunities_to_table(self, opportunities: List[PropertyOpportunity]):
        return [
            [
                opp.property.address,
                opp.property.property_type.title(),
                f"{opp.property.bedrooms}bd / {opp.property.bathrooms}ba",
                f"${opp.property.listing_price:,.0f}",
                f"${opp.estimated_value:,.0f}",
                f"${opp.value_gap:+,.0f}",
                f"{opp.investment_score:.0f} / 100",
                f"${opp.mortgage.monthly_payment:,.2f}/mo",
            ]
            for opp in opportunities
            if isinstance(opp, PropertyOpportunity)
        ]

    # ------------------------------------------------------------------
    # Scan workflow
    # ------------------------------------------------------------------
    def run_scan(self) -> List[PropertyOpportunity]:
        fw = self.get_framework()
        logging.info(
            f"Scan triggered at {datetime.now().strftime('%H:%M:%S')} | "
            f"Memory: {len(fw.memory)} properties"
        )
        fw.run()
        logging.info(f"Scan complete | Total analyzed: {len(fw.memory)}")
        return self.opportunities_to_table(fw.memory)

    def scan_with_logging(self, log_data):
        log_queue   = queue.Queue()
        result_queue = queue.Queue()
        setup_logging(log_queue)

        def worker():
            result = self.run_scan()
            result_queue.put(result)

        thread = threading.Thread(target=worker, daemon=True)
        thread.start()

        fw = self.get_framework()
        current_table = self.opportunities_to_table(fw.memory)

        while True:
            try:
                message = log_queue.get_nowait()
                log_data.append(reformat_log(message))
                current_table = self.opportunities_to_table(fw.memory)
                yield log_data, html_for(log_data), current_table, get_plot(fw)
            except queue.Empty:
                try:
                    final_table = result_queue.get_nowait()
                    yield log_data, html_for(log_data), final_table, get_plot(fw)
                    return
                except queue.Empty:
                    current_table = self.opportunities_to_table(fw.memory)
                    yield log_data, html_for(log_data), current_table, get_plot(fw)
                    time.sleep(0.1)

    # ------------------------------------------------------------------
    # Row selection -> detail panel
    # ------------------------------------------------------------------
    def handle_selection(self, selected_index: gr.SelectData) -> str:
        fw  = self.get_framework()
        row = selected_index.index[0]

        if row < len(fw.memory):
            opp = fw.memory[row]
            return opp.analysis_summary

        return "Select a row to view the full property analysis."

    # ------------------------------------------------------------------
    # Initial load
    # ------------------------------------------------------------------
    def load_initial_state(self):
        fw = self.get_framework()
        table = self.opportunities_to_table(fw.memory)
        return [], "", table, get_plot(fw)

    # ------------------------------------------------------------------
    # Mortgage calculator callback
    # ------------------------------------------------------------------
    def run_mortgage_calc(self, home_price: float, annual_income: float) -> str:
        if home_price <= 0:
            return "<p style='color:#dd4444'>Please enter a valid home price.</p>"
        return calculate_mortgage_scenarios(home_price, annual_income)

    # ------------------------------------------------------------------
    # Image analysis callback (multi-modal)
    # ------------------------------------------------------------------
    def analyze_uploaded_image(self, image, description: str, prop_type: str) -> str:
        if image is None and not description:
            return "Upload a property image or provide a description to analyze."

        agent = ImageAnalysisAgent()
        mock_prop = Property(
            address="User Uploaded Property",
            property_type=prop_type.lower(),
            bedrooms=3, bathrooms=2, sqft=1800,
            listing_price=400_000, year_built=2005,
            neighborhood="Unknown",
            description=description or "No description provided.",
            url="",
        )

        # If a real image is provided, in production this path would be sent to a vision API
        image_path = "uploaded" if image is not None else None
        vision_notes = agent.analyze_image(image_path, mock_prop)

        valuator = ValuationAgent()
        est = valuator.estimate(mock_prop)

        return (
            f"Property Type: {prop_type}\n"
            f"Description Provided: {description or 'None'}\n\n"
            f"Visual Analysis:\n{vision_notes}\n\n"
            f"Baseline Valuation Estimate: ${est:,.0f}\n"
            f"Note: This estimate uses heuristic rules. Upload to a production system for AVM-grade accuracy."
        )

    # ------------------------------------------------------------------
    # UI launch
    # ------------------------------------------------------------------
    def launch(self):
        with gr.Blocks(
            title="Real Estate and Mortgage Agent",
            fill_width=True,
            theme=gr.themes.Base(primary_hue="blue", neutral_hue="slate")
        ) as ui:

            log_data = gr.State([])

            gr.HTML(
                '<div style="text-align:center; padding:24px 0 8px;">'
                '<div style="font-size:26px; font-weight:700; color:#aaddff; letter-spacing:0.5px;">'
                'Real Estate and Mortgage Analysis System</div>'
                '<div style="font-size:14px; color:#666; margin-top:6px;">'
                'Multi-Agent | Multi-Modal | Automated Property Intelligence</div>'
                '</div>'
            )

            with gr.Tabs():

                # ---- Tab 1: Property Scanner ----
                with gr.Tab("Property Scanner"):

                    with gr.Row():
                        gr.Markdown(
                            "Properties are scanned automatically every 30 seconds. "
                            "Select a row to view the full analysis report below."
                        )

                    property_table = gr.Dataframe(
                        headers=[
                            "Address", "Type", "Beds/Baths",
                            "List Price", "Est. Value", "Value Gap",
                            "Inv. Score", "Monthly Pmt"
                        ],
                        wrap=True,
                        column_widths=[5, 1.5, 1.5, 1.5, 1.5, 1.5, 1.5, 1.5],
                        row_count=8,
                        col_count=8,
                        max_height=360,
                        interactive=False,
                    )

                    detail_box = gr.Textbox(
                        label="Property Analysis Detail",
                        lines=6,
                        interactive=False,
                        placeholder="Select a row above to view the full analysis report."
                    )

                    with gr.Row():
                        with gr.Column(scale=1):
                            logs_display = gr.HTML(label="Agent Activity Log")
                        with gr.Column(scale=1):
                            vector_plot = gr.Plot(value=None, show_label=False)

                # ---- Tab 2: Mortgage Calculator ----
                with gr.Tab("Mortgage Calculator"):
                    gr.Markdown("Compare mortgage scenarios across down payment amounts and loan terms.")

                    with gr.Row():
                        with gr.Column():
                            home_price_input  = gr.Number(label="Home Price ($)", value=450_000)
                            income_input      = gr.Number(label="Annual Household Income ($)", value=95_000)
                            calc_btn          = gr.Button("Calculate Scenarios")
                        with gr.Column(scale=2):
                            mortgage_output = gr.HTML(label="Mortgage Scenarios")

                    calc_btn.click(
                        self.run_mortgage_calc,
                        inputs=[home_price_input, income_input],
                        outputs=[mortgage_output]
                    )

                # ---- Tab 3: Image / Description Analysis (Multi-modal) ----
                with gr.Tab("Property Image Analysis"):
                    gr.Markdown(
                        "Upload a property photo and provide a description. "
                        "The vision agent will assess condition and the valuation agent will produce a baseline estimate. "
                        "In production, this tab connects to a GPT-4o or Claude vision API."
                    )

                    with gr.Row():
                        with gr.Column():
                            image_input   = gr.Image(label="Property Photo (optional)", type="pil")
                            desc_input    = gr.Textbox(label="Property Description", lines=4,
                                                       placeholder="3 bed / 2 bath ranch, updated kitchen, new roof...")
                            type_dropdown = gr.Dropdown(
                                label="Property Type",
                                choices=["Residential", "Condo", "Commercial", "Land"],
                                value="Residential"
                            )
                            analyze_btn   = gr.Button("Analyze Property")
                        with gr.Column():
                            vision_output = gr.Textbox(
                                label="Analysis Output", lines=14, interactive=False
                            )

                    analyze_btn.click(
                        self.analyze_uploaded_image,
                        inputs=[image_input, desc_input, type_dropdown],
                        outputs=[vision_output]
                    )

            # ------------------------------------------------------------------
            # Event wiring for scanner tab
            # ------------------------------------------------------------------
            ui.load(
                self.load_initial_state,
                inputs=[],
                outputs=[log_data, logs_display, property_table, vector_plot]
            )

            timer = gr.Timer(value=30, active=True)
            timer.tick(
                self.scan_with_logging,
                inputs=[log_data],
                outputs=[log_data, logs_display, property_table, vector_plot]
            )

            property_table.select(
                self.handle_selection,
                inputs=[],
                outputs=[detail_box]
            )

        ui.launch(share=True, inbrowser=True)

In [ ]:
app = RealEstateApp()
app.launch()